# MTSamples dataset exploration 

Looking at the raw MTSamples dataset before writing acceptance criteria for case ingestion. 

In [1]:
import pandas as pd

In [7]:
df = pd.read_csv("../data/mtsamples.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4999 entries, 0 to 4998
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Unnamed: 0         4999 non-null   int64
 1   description        4999 non-null   str  
 2   medical_specialty  4999 non-null   str  
 3   sample_name        4999 non-null   str  
 4   transcription      4966 non-null   str  
 5   keywords           3931 non-null   str  
dtypes: int64(1), str(5)
memory usage: 234.5 KB


- Entries: 4999
- Columns: 6 | (Unnamed, description, medical_specialty, sample_name, transcription, keywords)
- Missing values: 
    - transcription 4999 - 4966 = 33
    - keywords 4999 - 3931 = 1068

In [8]:
df.head()

,Unnamed: 0,description,medical_specialty,sample_name,transcription,keywords
0,0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr...","allergy / immunology, allergic rhinitis, aller..."
1,1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb...","bariatrics, laparoscopic gastric bypass, weigh..."
2,2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ...","bariatrics, laparoscopic gastric bypass, heart..."
3,3,2-D M-Mode. Doppler.,Cardiovascular / Pulmonary,2-D Echocardiogram - 1,"2-D M-MODE: , ,1. Left atrial enlargement wit...","cardiovascular / pulmonary, 2-d m-mode, dopple..."
4,4,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 2,1. The left ventricular cavity size and wall ...,"cardiovascular / pulmonary, 2-d, doppler, echo..."


In [9]:
# Unnamed: 0 looks like a leftover index from how the csv was saved. 
(df["Unnamed: 0"].values == df.index.values).all()

np.True_

In [10]:
# Renaming "Unnamed 0" and dropping "keywords"
df = (
    df.rename(columns={"Unnamed: 0": "case_id"})
      .drop(columns=["keywords"])
)

df.head()

,case_id,description,medical_specialty,sample_name,transcription
0,0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr..."
1,1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb..."
2,2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ..."
3,3,2-D M-Mode. Doppler.,Cardiovascular / Pulmonary,2-D Echocardiogram - 1,"2-D M-MODE: , ,1. Left atrial enlargement wit..."
4,4,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 2,1. The left ventricular cavity size and wall ...


In [11]:
# Checking missing values in the rest of the dataset, more specifically in the `transcription` column
df.isna().sum()

case_id               0
description           0
medical_specialty     0
sample_name           0
transcription        33
dtype: int64

In [12]:
# Rows with missing transcriptions can't be used in this project
df = df.dropna(subset=["transcription"])

In [13]:
df.shape

(4966, 5)

Comparing description against transcription to confirm which column is the actual clincal note.

In [14]:
df["description"].iloc[0]

' A 23-year-old white female presents with complaint of allergies.'

In [15]:
df["transcription"].iloc[0]

'SUBJECTIVE:,  This 23-year-old white female presents with complaint of allergies.  She used to have allergies when she lived in Seattle but she thinks they are worse here.  In the past, she has tried Claritin, and Zyrtec.  Both worked for short time but then seemed to lose effectiveness.  She has used Allegra also.  She used that last summer and she began using it again two weeks ago.  It does not appear to be working very well.  She has used over-the-counter sprays but no prescription nasal sprays.  She does have asthma but doest not require daily medication for this and does not think it is flaring up.,MEDICATIONS: , Her only medication currently is Ortho Tri-Cyclen and the Allegra.,ALLERGIES: , She has no known medicine allergies.,OBJECTIVE:,Vitals:  Weight was 130 pounds and blood pressure 124/78.,HEENT:  Her throat was mildly erythematous without exudate.  Nasal mucosa was erythematous and swollen.  Only clear drainage was seen.  TMs were clear.,Neck:  Supple without adenopathy.,

In [17]:
# Sampling over a few random transcriptions to examine note structure, use of abbreviations, acronyms, inconsistent formatting across different notes. 
for text in df["transcription"].sample(20, random_state=42):
    print(text)
    print("---")

HISTORY OF PRESENT ILLNESS:,  The patient is well known to me for a history of iron-deficiency anemia due to chronic blood loss from colitis.  We corrected her hematocrit last year with intravenous (IV) iron.  Ultimately, she had a total proctocolectomy done on 03/14/2007 to treat her colitis.  Her course has been very complicated since then with needing multiple surgeries for removal of hematoma.  This is partly because she was on anticoagulation for a right arm deep venous thrombosis (DVT) she had early this year, complicated by septic phlebitis.,Chart was reviewed, and I will not reiterate her complex history.,I am asked to see the patient again because of concerns for coagulopathy.,She had surgery again last month to evacuate a pelvic hematoma, and was found to have vancomycin resistant enterococcus, for which she is on multiple antibiotics and followed by infectious disease now.,She is on total parenteral nutrition (TPN) as well.,LABORATORY DATA:,  Labs today showed a white blood 

Across a sample of 20 transcriptions, every note starts with an ALL-CAPS section header followed by a colon and comma (e.g., HISTORY OF PRESENT ILLNESS:,), though spacing around the comma is inconsistent (:, vs : ,). Section names vary by note type, an operative note and a routine visit note use different sections, which is expected. Sections are separated by commas rather than newlines throughout. 3 of 20 notes contain ___ placeholders, likely redacted or illegible content, recurring often enough to document as a real limitation. Abbreviations and acronyms are common (TPN, PT, INR, PTT, LFTs, DVT, p.r.n.), consistent with real clinical dictation.

In [18]:
# Checking the `medical_specialty` column to see note transcription distribution. 
df["medical_specialty"].value_counts()

medical_specialty
Surgery                          1088
Consult - History and Phy.        516
Cardiovascular / Pulmonary        371
Orthopedic                        355
Radiology                         273
General Medicine                  259
Gastroenterology                  224
Neurology                         223
SOAP / Chart / Progress Notes     166
Urology                           156
Obstetrics / Gynecology           155
Discharge Summary                 108
ENT - Otolaryngology               96
Neurosurgery                       94
Hematology - Oncology              90
Ophthalmology                      83
Nephrology                         81
Emergency Room Reports             75
Pediatrics - Neonatal              70
Pain Management                    61
Psychiatry / Psychology            53
Office Notes                       50
Podiatry                           47
Dermatology                        29
Dentistry                          27
Cosmetic / Plastic Surgery      

The `medical_specialty` column shows Surgery as the dominant specialty with 1088 notes, while roughly a dozen specialties have fewer than 20 notes each, five with fewer than 10. Some entries in this column aren't actual medical specialties, but note types, SOAP / Chart / Progress Notes, Letters, Consult - History and Phy., and Office Notes among them. Anything that filters or samples by specialty needs to account for this mix, not treat every value as a real specialty.

The working dataset, after cleaning, will consist of 4966 rows, 5 fields (`case_id`, `description`, `medical_specialty`, `sample_name`, `transcription`). When ingesting the dataset it must drop `keywords` entirely, drop rows missing `transcription`, use `transcription` as the case text, and use the remaining fields as metadata. See above for the specialty/document-type mix, the comma-delimited header formatting, and the `___` placeholder pattern.